# 03 · Test

Calcola i punteggi del modello migliore di ogni seed sulle liste di test e li salva in `punteggi_<lista>.csv`
nella cartella dell'esperimento. I test già fatti vengono saltati; le liste lunghe si fanno a blocchi
e, se si interrompe, si riparte dal blocco dove era arrivato.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))   # per importare la cartella tesi/ del progetto

import pandas as pd

from tesi import training, metriche

## 1. Test sui sottoinsiemi

Liste disponibili (create nel notebook 01):
- `asvspoof5_eval` (10.000 clip), `asvspoof2019_eval` (71.237), `inthewild` (31.779)
- `asvspoof2021la_eval_10k`, `asvspoof2021df_eval_10k` (per i modelli allenati su 2019 LA)

A circa 50 clip al secondo, 10.000 clip richiedono 3-4 minuti e 2019 LA eval circa 25 minuti.

In [ ]:
MODELLO = "mhfa"
DATASET = "asvspoof5"
AUGMENTATION = "rawboost"
SEEDS = [1, 2, 3]
TEST = ["asvspoof5_eval", "asvspoof2019_eval", "inthewild"]
if DATASET == "asvspoof2019":
    TEST += ["asvspoof2021la_eval_10k", "asvspoof2021df_eval_10k"]

In [ ]:
righe = []
for seed in SEEDS:
    cartella = training.cartella_esperimento(MODELLO, DATASET, AUGMENTATION, seed)
    for nome in TEST:
        p = training.testa(cartella, nome)
        righe.append({"seed": seed, "test": nome, **metriche.tutte((p["label"] == "spoof").astype(int), p["punteggio"])})
pd.DataFrame(righe).round(3)

## 2. Eval completi (solo seed 1 dei modelli principali)

ASVspoof 5 eval completo (circa 680.000 clip, ~3,5 ore), ASVspoof 2021 LA (~1 ora) e DF (~3 ore).
Servono per confrontare i numeri con i paper. Lanciali solo quando la GPU è libera.

In [ ]:
TEST_COMPLETI = ["asvspoof5_eval_completo"]
if DATASET == "asvspoof2019":
    TEST_COMPLETI = ["asvspoof2021la_eval", "asvspoof2021df_eval"]

cartella = training.cartella_esperimento(MODELLO, DATASET, AUGMENTATION, 1)
for nome in TEST_COMPLETI:
    p = training.testa(cartella, nome)
    print(nome, metriche.tutte((p["label"] == "spoof").astype(int), p["punteggio"]))